In this code we estimate the state transition equations for $R_t$ and $P_t$ and try and replicate the  the paper’s estimates. The estiamtes from the paper can be seen below:
$$
\begin{array}{lc}
\hline
\textbf{Statistic} & \textbf{Target Value} \\
\hline
\text{Avg. Production Cost }(c) & 231\ (\text{Real2000})  \\
\text{Residual Correlation} & 0.054 \\
\text{Residual Covariance} & 0.000886 \\
\text{Price Splicing Ratio} & 0.868  \\
\ln R_t \text{ Lag Coefficient} & 0.742391 \\
\hline
\end{array}
$$

In [1]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [3]:
folder = '/Users/mikkelEngelsted/Documents/KU/Cand.polit/Dynamic Programming/Term paper/Dynamic-Programming-Project/Data'
filename = 'Lohano_King_Data_Cleaned.csv'

filepath = os.path.join(folder, filename)
df = pd.read_csv(filepath, sep=';')
df.columns = df.columns.str.strip()

print("Columns:")
print(df.columns.tolist())

Columns:
['Year', 'Gross Return (nominal)', 'Costs (nominal)', 'Gross Return (nominal).1', 'Costs (nominal).1', 'Unnamed: 5', 'Gross Return (nominal).2', 'Costs (nominal).2', 'Gross Return (real) in 2000 dollars', 'Costs (real) in 2000 dollars', 'P: Raup (nominal)', 'P: Taff (nominal)', 'Ratio', 'P  (nominal)', 'P  (real) in 2000 dollars', 'Implicit Price Deflator of GNP', 'RM (percent) (Real)', '(1+rate of return): (Real)']


In [4]:
df = df.rename(columns={
    'Gross Return (real) in 2000 dollars': 'Rt',
    'Costs (real) in 2000 dollars': 'Ct',
    'P  (real) in 2000 dollars': 'Pt',
    '(1+rate of return): (Real)': 'Mt',
    'P: Raup (nominal)': 'P_raup',
    'P: Taff (nominal)': 'P_taff'
})

In [5]:
# Convert to numeric
for col in ['Year', 'Rt', 'Ct', 'Pt', 'Mt', 'P_raup', 'P_taff']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Sort once by year
df = df.sort_values('Year').reset_index(drop=True)

#### 1. RETURN EQUATION 
$$ ln(R_t) = beta0 + beta1 * ln(R_{t-1}) + error_t $$


In [6]:
# 1. RETURN EQUATION 
df_R = df[['Year', 'Rt']].copy()
df_R = df_R.dropna(subset=['Year', 'Rt'])
df_R = df_R[df_R['Rt'] > 0].copy()
df_R = df_R.sort_values('Year').reset_index(drop=True)

df_R['ln_Rt'] = np.log(df_R['Rt'])
df_R['ln_Rt_lag'] = df_R['ln_Rt'].shift(1)

# Keep only rows where lag exists
df_R_reg = df_R.dropna(subset=['ln_Rt', 'ln_Rt_lag']).copy()

X_R = sm.add_constant(df_R_reg['ln_Rt_lag'])
y_R = df_R_reg['ln_Rt']

model_R = sm.OLS(y_R, X_R).fit()
eps1 = model_R.resid

print("Return equation years used:",
      int(df_R_reg['Year'].min()), "-", int(df_R_reg['Year'].max()))
print("Return equation observations used:", len(df_R_reg))

Return equation years used: 1968 - 2007
Return equation observations used: 40


#### 2. PRICE EQUATION
This equation should be estimated on rows where Rt and Pt exist:
$$\ln P_t = \alpha_0 + \alpha_1 \ln P_{t-1} + \alpha_2 \ln P_{t-2} + \alpha_3 \ln R_{t-1} + \varepsilon_{2t}$$

In [12]:
# 2. PRICE EQUATION
df_P = df[['Year', 'Rt', 'Pt']].copy()
df_P = df_P.dropna(subset=['Year', 'Rt', 'Pt'])
df_P = df_P[(df_P['Rt'] > 0) & (df_P['Pt'] > 0)].copy()
df_P = df_P.sort_values('Year').reset_index(drop=True)

df_P['ln_Rt'] = np.log(df_P['Rt'])
df_P['ln_Pt'] = np.log(df_P['Pt'])
df_P['ln_Pt_lag1'] = df_P['ln_Pt'].shift(1)
df_P['ln_Pt_lag2'] = df_P['ln_Pt'].shift(2)
df_P['ln_Rt_lag1'] = df_P['ln_Rt'].shift(1)

df_P_reg = df_P.dropna(subset=['ln_Pt', 'ln_Pt_lag1', 'ln_Pt_lag2', 'ln_Rt_lag1']).copy()

X_P = sm.add_constant(df_P_reg[['ln_Pt_lag1', 'ln_Pt_lag2', 'ln_Rt_lag1']])
y_P = df_P_reg['ln_Pt']

model_P = sm.OLS(y_P, X_P).fit()
eps2 = model_P.resid

print("Price equation years used:",
      int(df_P_reg['Year'].min()), "-", int(df_P_reg['Year'].max()))
print("Price equation observations used:", len(df_P_reg))

Price equation years used: 1969 - 2007
Price equation observations used: 39


#### 3. MUTUAL FUND EQUATION

$$
\ln M_{t+1} = \gamma_0+\epsilon_{3,t+1}
$$

In [52]:
# 3. MUTUAL FUND EQUATION
df_M = df[['Year', 'Mt']].copy()
df_M = df_M.dropna(subset=['Year', 'Mt']).copy()
df_M = df_M[df_M['Mt'] > 0].copy()
df_M = df_M.sort_values('Year').reset_index(drop=True)

df_M['ln_Mt'] = np.log(df_M['Mt'])

df_M_reg = df_M.dropna(subset=['ln_Mt']).copy()

y_M = df_M_reg['ln_Mt']
X_M = np.ones((len(y_M), 1))  # Only constant, no independent variables

model_M = sm.OLS(y_M, X_M).fit()
eps3 = model_M.resid

print("Mutual fund equation years used:",
      int(df_M_reg['Year'].min()), "-", int(df_M_reg['Year'].max()))
print("Mutual fund equation observations used:", len(df_M_reg))

Mutual fund equation years used: 1967 - 2007
Mutual fund equation observations used: 41


#### 4. RESIDUAL COVARIANCE AND CORRELATION

In [53]:
# 4. RESIDUAL COVARIANCE AND CORRELATION
eps_df = pd.DataFrame({
    'eps1': eps1,
    'eps2': eps2,
    'eps3': eps3
}).dropna()

residual_cov = eps_df['eps1'].cov(eps_df['eps2'])
residual_corr = eps_df['eps1'].corr(eps_df['eps2'])

price_residual_var = eps2.var()
return_residual_var = eps1.var()

price_residual_var = model_P.mse_resid  # Mean squared error of residuals
return_residual_var = model_R.mse_resid


#### 5. AVERAGE PRODUCTION COST
Paper uses average real production cost + 30 for labor/management. 

Costs are available only from later years, so this is a separate calculation.

In [54]:
# 5. AVERAGE PRODUCTION COST
df_C = df[['Year', 'Ct']].copy()
df_C = df_C.dropna(subset=['Year', 'Ct']).copy()
df_C = df_C[df_C['Ct'] > 0].copy()

avg_cost_data = df_C['Ct'].mean()
c_total = avg_cost_data + 30

print("Cost years used:",
      int(df_C['Year'].min()), "-", int(df_C['Year'].max()))
print("Cost observations used:", len(df_C))

Cost years used: 1983 - 2007
Cost observations used: 25


In [ ]:
# Parameters from the paper
kappa = 300  # Machinery and equipment per acre
rho = 0.70   # Maximum debt-to-asset ratio
r_borrow = 0.06  # Interest rate on borrowing
r_lend = 0.03    # Interest rate on lending (risk-free)

# Transaction costs
tc_buy_land = 0.01
tc_sell_land = 0.06
tc_sell_machinery = 0.07

=== Model Parameters ===
Machinery cost (κ): $300
Production cost (c): $231
Max debt-to-asset ratio (ρ): 0.7
Borrowing rate: 6.0%
Lending rate: 3.0%


#### 6. PRICE SPLICING RATIO
Ratio of Taff to Raup over overlap years


In [55]:
df_ratio = df[['Year', 'P_raup', 'P_taff']].copy()
df_ratio = df_ratio.dropna(subset=['P_raup', 'P_taff']).copy()
df_ratio = df_ratio[(df_ratio['P_raup'] > 0) & (df_ratio['P_taff'] > 0)].copy()

df_ratio['splice_ratio'] = df_ratio['P_taff'] / df_ratio['P_raup']
price_splice = df_ratio['splice_ratio'].mean()

print("Splicing overlap years used:",
      int(df_ratio['Year'].min()), "-", int(df_ratio['Year'].max()))
print("Splicing observations used:", len(df_ratio))

Splicing overlap years used: 1990 - 1992
Splicing observations used: 3


#### 7. RESULTS

In [ ]:
# 6. RESULTS
print('\n--- Estimated coefficients ---')
print('\nReturn equation:')
print(model_R.params)
print(f'Variance of return residuals: {return_residual_var:.6f}')

print('\nPrice equation:')
print(model_P.params)
print(f'Variance of price residuals:  {price_residual_var:.6f}')

print('\nMutual fund equation:')
print(model_M.params)
print(f'Variance of mutual fund residuals: {eps3.var():.6f}')

print('\n--- Summary statistics ---')
print(f'Average production cost from data: {avg_cost_data:.6f}')
print(f'Total cost parameter c (+30):      {c_total:.6f}')
print(f'Residual correlation:              {residual_corr:.6f}')
print(f'Residual covariance:               {residual_cov:.6f}')
print(f'Price splicing ratio:              {price_splice:.6f}')
print(f'Return lag coefficient:            {model_R.params["ln_Rt_lag"]:.6f}')


print('\n--- Return equation summary ---')
print(model_R.summary())

print('\n--- Price equation summary ---')
print(model_P.summary())

print('\n--- Mutual fund equation summary ---')
print(model_M.summary())


--- Estimated coefficients ---

Return equation:
const        1.511810
ln_Rt_lag    0.742391
dtype: float64
Variance of return residuals: 0.030186

Price equation:
const         0.353963
ln_Pt_lag1    1.495469
ln_Pt_lag2   -0.646337
ln_Rt_lag1    0.130772
dtype: float64
Variance of price residuals:  0.010068

Mutual fund equation:
const    0.057757
dtype: float64
Variance of mutual fund residuals: 0.026941

--- Summary statistics ---
Average production cost from data: 201.094000
Total cost parameter c (+30):      231.094000
Residual correlation:              0.054393
Residual covariance:               0.000910
Price splicing ratio:              0.867253
Return lag coefficient:            0.742391

--- Return equation summary ---
                            OLS Regression Results                            
Dep. Variable:                  ln_Rt   R-squared:                       0.533
Model:                            OLS   Adj. R-squared:                  0.521
Method:                

In [32]:
# Reduced price equation with only one lag of price and return
alpha0 = model_P.params['const'] / (1 - model_P.params['ln_Pt_lag2'])
alpha1 = model_P.params['ln_Pt_lag1'] / (1 - model_P.params['ln_Pt_lag2'])
alpha2 = model_P.params['ln_Rt_lag1'] / (1 - model_P.params['ln_Pt_lag2'])
epsilon_t = price_residual_var / (1 - (model_P.params['ln_Pt_lag2'])**2)

In [35]:
print('\n--- Reduced price equation coefficients ---')
print(f'alpha0: {alpha0:.6f}')
print(f'alpha1: {alpha1:.6f}')
print(f'alpha2: {alpha2:.6f}')
print(f'epsilon_t variance: {epsilon_t:.6f}')


--- Reduced price equation coefficients ---
alpha0: 0.215001
alpha1: 0.908361
alpha2: 0.079432
epsilon_t variance: 0.017292
